# EPLAN Electric P8 — Fine-Tune with QLoRA (No Unsloth)

Fine-tunes **Qwen 2.5 3B Instruct** on 4,288 Q&A pairs from EPLAN documentation.

**Stack:** Transformers + TRL + PEFT + BitsAndBytes (native HuggingFace, no Unsloth)
**Runtime:** Kaggle GPU T4 x2 (free tier)
**Time:** ~1-2 hours

## Steps
1. Install dependencies
2. Load model with 4-bit quantization (QLoRA)
3. Configure LoRA adapters
4. Load & format EPLAN dataset
5. Train with SFTTrainer
6. Test the model
7. Merge LoRA + push to HuggingFace Hub
8. Convert to GGUF for Ollama

## 1. Install Dependencies

In [ ]:
%%capture install_log
# Step 1: Check pre-installed torch version and reinstall matching torchvision
import torch
tv = f"torchvision=={'.'.join(torch.__version__.split('.')[:2])+'.*'}"

import subprocess, sys

# Reinstall torchvision matching Kaggle's torch (fixes nms operator error)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torchvision", "--quiet"])

# Install HuggingFace stack — pin transformers to avoid torchvision import bugs
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U",
    "transformers==4.51.3",
    "datasets>=3.3.0",
    "accelerate>=1.4.0",
    "bitsandbytes>=0.45.0",
    "trl==0.16.1",
    "peft==0.15.2",
    "sentencepiece",
    "protobuf<4",
    "--quiet"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy<2", "--quiet"])

# Restart runtime to load packages cleanly
import os
os._exit(0)

In [ ]:
# Verify installations after kernel restart
import numpy, torch, transformers, peft, trl, bitsandbytes
print(f"numpy:          {numpy.__version__}")
print(f"torch:          {torch.__version__}")
print(f"transformers:   {transformers.__version__}")
print(f"peft:           {peft.__version__}")
print(f"trl:            {trl.__version__}")
print(f"bitsandbytes:   {bitsandbytes.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name() if torch.cuda.is_available() else 'N/A'}")

# Login to HuggingFace (needed for gated models like Gemma)
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("HuggingFace: logged in")

## 2. Load Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Qwen 2.5 3B Instruct — text-only, no gated license, fits T4 easily
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 2048

# T4 uses fp16
torch_dtype = torch.float16

print(f"GPU 0: {torch.cuda.get_device_name(0)}")
print(f"GPU 1: {torch.cuda.get_device_name(1)}")
print(f"Precision: {torch_dtype}")

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
)

# Spread model across both T4 GPUs
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch_dtype,
    device_map="balanced",
    attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device map: {model.hf_device_map}")

## 3. Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training (skip float32 upcast to save VRAM)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 4. Load & Format Dataset

In [ ]:
from datasets import load_dataset

# Load from HuggingFace Hub
dataset = load_dataset("covaga/eplan-qa-dataset", data_files="eplan_qa_FINAL.jsonl", split="train")

print(f"Dataset: {len(dataset)} examples")
print(f"Columns: {dataset.column_names}")
print(dataset[0])

In [ ]:
SYSTEM_MSG = """You are an expert EPLAN Electric P8 assistant specialized in industrial electrical engineering. 
You help engineers with API usage, troubleshooting, best practices, and procedural guidance. 
Provide accurate, detailed answers based on EPLAN documentation."""

def format_chat(example):
    """Convert instruction/output to Qwen chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"messages": messages}

dataset = dataset.map(format_chat, remove_columns=["source", "category"])

# Verify format
print("Sample formatted:")
sample_text = tokenizer.apply_chat_template(
    dataset[0]["messages"], tokenize=False, add_generation_prompt=False
)
print(sample_text[:600])

## 5. Train

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="eplan-qwen-qlora",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    num_train_epochs=3,
    per_device_train_batch_size=1,       # Reduced from 2
    gradient_accumulation_steps=8,       # Effective batch = 8
    gradient_checkpointing=True,
    optim="adamw_8bit",                  # 8-bit Adam uses less VRAM than fused
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=(torch_dtype == torch.float16),
    bf16=(torch_dtype == torch.bfloat16),
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    weight_decay=0.01,
    seed=42,
    report_to="none",
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": True,
    },
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting training...")
stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Loss: {stats.training_loss:.4f}")
print(f"  Runtime: {stats.metrics['train_runtime']:.0f}s")

trainer.save_model()

## 5b. Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt

# Extract loss from trainer logs
log_history = trainer.state.log_history
steps = [x["step"] for x in log_history if "loss" in x]
losses = [x["loss"] for x in log_history if "loss" in x]

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, "b-o", markersize=4, label="Train Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Loss Curve — EPLAN Fine-Tune (Qwen 2.5 3B, QLoRA)")
plt.legend()
plt.grid(True, alpha=0.3)

# Mark epoch boundaries
total_steps = steps[-1] if steps else 0
for epoch in range(1, 4):
    epoch_step = int(total_steps * epoch / 3)
    if epoch_step <= total_steps:
        plt.axvline(x=epoch_step, color="r", linestyle="--", alpha=0.5, label=f"Epoch {epoch}" if epoch == 1 else "")

plt.tight_layout()
plt.savefig("training_loss.png", dpi=150)
plt.show()
print(f"Min loss: {min(losses):.4f} at step {steps[losses.index(min(losses))]}")
print(f"Final loss: {losses[-1]:.4f}")

## 6. Test the Model

In [ ]:
from transformers import pipeline

# Free training memory
del trainer
torch.cuda.empty_cache()

# Test with pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

test_questions = [
    "How do I export a project to PDF using the EPLAN API?",
    "What is the difference between a Function and a FunctionBase?",
    "My script throws NullReferenceException when iterating pages. What could cause this?",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": q},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = pipe(
        prompt,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")],
    )
    response = outputs[0]["generated_text"][len(prompt):].strip()
    print(f"Q: {q}")
    print(f"A: {response}")
    print("-" * 80)

## 7. Push to HuggingFace Hub

Already logged in from Step 1.

## 8. Merge LoRA + Push

In [ ]:
# Push LoRA adapter first (small, ~100MB)
model.push_to_hub("covaga/eplan-assistant-lora")
tokenizer.push_to_hub("covaga/eplan-assistant-lora")
print("LoRA adapter pushed!")

In [ ]:
from peft import PeftModel

# Free memory
del model, pipe
torch.cuda.empty_cache()

# Reload base model in full precision for merging
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto",
    attn_implementation="eager",
    low_cpu_mem_usage=True,
)

# Load and merge LoRA weights
merged_model = PeftModel.from_pretrained(base_model, "eplan-gemma-qlora")
merged_model = merged_model.merge_and_unload()

# Save merged model locally
merged_model.save_pretrained("eplan-assistant-merged", safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained("eplan-assistant-merged")
print("Merged model saved locally!")

# Push merged model to Hub
merged_model.push_to_hub("covaga/eplan-assistant-merged")
tokenizer.push_to_hub("covaga/eplan-assistant-merged")
print("Merged model pushed to HuggingFace!")

from peft import PeftModel

# Free memory
del model, pipe
torch.cuda.empty_cache()

# Reload base model in full precision for merging
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto",
    attn_implementation="eager",
    low_cpu_mem_usage=True,
)

# Load and merge LoRA weights
merged_model = PeftModel.from_pretrained(base_model, "eplan-qwen-qlora")
merged_model = merged_model.merge_and_unload()

# Save merged model locally
merged_model.save_pretrained("eplan-assistant-merged", safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained("eplan-assistant-merged")
print("Merged model saved locally!")

# Push merged model to Hub
merged_model.push_to_hub("covaga/eplan-assistant-merged")
tokenizer.push_to_hub("covaga/eplan-assistant-merged")
print("Merged model pushed to HuggingFace!")